# ДЗ 6. Очистка orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw06` репозитория `dsa-homework` (`hw06.ipynb`): `git add hw06` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: всё с уроков 1–6. `groupby`, `pivot_table`, `merge` пока не используем (урок 7). Небольшой цикл по городам или месяцам допустим. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([1, None, 3])
print(s.isna().sum(), s.fillna(0).mean())
# 2
print(pd.Series([" a", "A ", "b"]).str.strip().str.lower().nunique())
# 3
df = pd.DataFrame({"k": [1, 1, 2]})
df.drop_duplicates()
print(len(df))
# 4
print(pd.to_numeric(pd.Series(["1 000", "5"]).str.replace(" ", "")).sum())
# 5
d = pd.to_datetime(pd.Series(["2026-09-30 21:00", "2026-10-01 08:00"]))
print(d.dt.month.tolist())
```

Ваши ответы и объяснения:

1. 1 1.3333 isna().sum() находит один пропуск
2. 2 
3. 3
4. 1005
5. [9, 10]

In [1]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


## [База] 1. Функция очистки
Напишите функцию `clean(path)`, которая читает orders.csv и делает все шаги урока: суммы в числа, города через словарь, дубли, даты, без TEST. Отменённые заказы **не** удаляйте: они нужны для доли отмен.
После каждого шага функция печатает, сколько строк осталось. Проверки внутри функции: у `amount` нет NaN, у `city` нет NaN, `order_id` уникален.
Сколько строк в результате?

In [3]:
import pandas as pd

def clean(path):
    df = pd.read_csv(path)
    print("Исходных строк:", len(df))

    df["amount"] = pd.to_numeric(
        df["amount"].astype(str).str.replace(" ", "", regex=False)
    )
    print("После обработки сумм:", len(df))

    cities = {
        "almaty": "Алматы",
        "astana": "Астана",
        "shymkent": "Шымкент"
    }

    df["city"] = df["city"].str.strip().str.lower()
    df["city"] = df["city"].replace(cities).str.title()
    print("После очистки городов:", len(df))

    df = df.drop_duplicates(subset="order_id").copy()
    print("После удаления дублей:", len(df))

    df["order_date"] = pd.to_datetime(df["order_date"])
    print("После обработки дат:", len(df))

    df = df[df["customer_id"] != "TEST"].copy()
    print("После удаления TEST:", len(df))

    assert df["amount"].notna().all()
    assert df["city"].notna().all()
    assert df["order_id"].is_unique

    return df


df = clean("orders.csv")

print("\nИтог:", len(df), "строк")

Исходных строк: 5463
После обработки сумм: 5463
После очистки городов: 5463
После удаления дублей: 5283
После обработки дат: 5283
После удаления TEST: 5280

Итог: 5280 строк


## [База] 2. Выручка по чистым данным
Выручка оплаченных заказов за сентябрь и октябрь и изменение в %. Затем то же для каждого из 6 городов (цикл по городам).
Совпадает ли с таблицей «город × месяц» урока 4? Какой город падает сильнее всех?

In [6]:
# 2
paid = df[df["status"] == "оплачен"].copy()

sep = paid["order_date"].dt.month == 9
octo = paid["order_date"].dt.month == 10

sep_revenue = paid.loc[sep, "amount"].sum()
oct_revenue = paid.loc[octo, "amount"].sum()

change = (oct_revenue - sep_revenue) / sep_revenue * 100

print(f"Сентябрь: {sep_revenue:.0f} ₸")
print(f"Октябрь: {oct_revenue:.0f} ₸")
print(f"Изменение: {change:.1f}%")

print("\nВыручка по городам:")

for city in sorted(df["city"].unique()):
    city_orders = paid[paid["city"] == city]

    s = city_orders.loc[
        city_orders["order_date"].dt.month == 9, "amount"
    ].sum()

    o = city_orders.loc[
        city_orders["order_date"].dt.month == 10, "amount"
    ].sum()

    percent = (o - s) / s * 100

    print(f"{city:<12} | {s:>12.0f} | {o:>12.0f} | {percent:>7.1f}%")

Сентябрь: 35243410 ₸
Октябрь: 34419520 ₸
Изменение: -2.3%

Выручка по городам:
Актобе       |      2576960 |      2540420 |    -1.4%
Алматы       |     14501070 |     14643020 |     1.0%
Астана       |      9034220 |      8850530 |    -2.0%
Атырау       |       923990 |       969860 |     5.0%
Караганда    |      3648520 |      3371680 |    -7.6%
Шымкент      |      4558650 |      4044010 |   -11.3%


## [База] 3. Журнал очистки для менеджера
Markdown-таблица: шаг очистки → сколько строк убрал → как изменилось «падение выручки» (от −22,7% до −2,3%).
Одной фразой для менеджера: какой шаг сильнее всего исказил его цифру?

In [ ]:
# 3 результат исказили заказы клиента TEST

## [Стандарт] 4. Пропуски в сроках доставки
Сколько пропусков в `delivery_days` по чистым данным? В каких городах и статусах их больше?
Доля доставки за 2 дня в Шымкенте: (а) `dropna`, (б) `fillna(медиана)`, (в) `fillna(0)`. Какой вариант честный и почему другие вводят в заблуждение?

In [5]:
# 4
missing = df["delivery_days"].isna()

print("Всего пропусков:", missing.sum())

print("\nПропуски по городам:")
print(df.loc[missing, "city"].value_counts())

print("\nПропуски по статусам:")
print(df.loc[missing, "status"].value_counts())

sh = df[df["city"] == "Шымкент"].copy()

drop = sh["delivery_days"].dropna()
median = sh["delivery_days"].median()

a = (drop == 2).mean() * 100
b = (sh["delivery_days"].fillna(median) == 2).mean() * 100
c = (sh["delivery_days"].fillna(0) == 2).mean() * 100

print("\nДоля доставки за 2 дня в Шымкенте:")
print(f"dropna: {a:.1f}%")
print(f"fillna(медиана): {b:.1f}%")
print(f"fillna(0): {c:.1f}%")

Всего пропусков: 158

Пропуски по городам:
city
Алматы       63
Астана       40
Караганда    24
Шымкент      22
Атырау        5
Актобе        4
Name: count, dtype: int64

Пропуски по статусам:
status
оплачен    144
отменён     14
Name: count, dtype: int64

Доля доставки за 2 дня в Шымкенте:
dropna: 14.9%
fillna(медиана): 14.4%
fillna(0): 14.4%


## [Стандарт] 5. Исправьте код от AI
AI написал «очистку» и получил 10 городов вместо 6, а строк осталось 5 463. В коде **две** ошибки. Найдите их и исправьте.
```python
df = pd.read_csv("orders.csv")
df.drop_duplicates()
df["city"] = df["city"].str.lower()
print(df["city"].nunique(), len(df))
```

In [ ]:
# 5

## [Стандарт] 6. Дни недели
По чистым данным: сколько заказов в каждый день недели (`.dt.day_name()` и `value_counts`)? В какой день заказов меньше всего и на сколько процентов меньше, чем в самый загруженный?
Что это значит для графика работы курьеров?

In [ ]:
# 6

## [Челлендж] 7. Когда сломалась доставка в Шымкенте
Средний срок доставки в Шымкенте в сентябре и в октябре. Найдите первую дату, с которой срок доставки в Шымкенте не бывает меньше 5 дней (фильтр + `sort_values` по дате).
Проверьте, что в других городах в эту дату ничего не изменилось. Напишите менеджеру: что случилось, когда и что проверить в первую очередь.

In [ ]:
# 7

## [Челлендж] 8. Надёжный перевод сумм
Напишите функцию `to_amount(s)` для Series строк, которая понимает `"15 990"`, `"15990.0"`, `"1 200,50"` (запятая как разделитель дробной части) и превращает всё остальное (`"нет данных"`) в NaN.
Проверьте на `pd.Series(["15 990", "15990.0", "1 200,50", "нет данных"])` и на столбце `amount` из orders.csv: сумма должна совпасть с уроком.

In [ ]:
# 8